# 3.4 Slide-Enhanced Ceiling Analysis

This notebook extends the annotation-ceiling analysis with methods motivated by annotation biology and the project design. The key premise is that *disorder is not one single ground-truth property*: NMR chemical shifts, curated binary disorder, AlphaFold confidence, structural flexibility/B-values, and derived soft labels each capture different biological facets.

The goal is therefore not only to draw ceiling plots, but to interpret when low agreement is a technical limitation, when it is a biological definition mismatch, and when a cross-dataset transfer result is plausible.


## Added Methods

This notebook adds the following measurement-aware analyses:

1. Annotation-family grouping by biological meaning.
2. Expected-vs-observed biological compatibility scoring.
3. Overlap-weighted confidence labels.
4. Same-concept vs cross-concept ceiling summaries.
5. Direction-specific transfer interpretation.
6. Threshold sensitivity for comparisons involving DisProt.
7. Region-level agreement after thresholding continuous labels.
8. PDBFlex as a flexibility-not-disorder contrast/control.
9. A final biological claim table tying numerical evidence to biological measurement concepts.


In [ ]:
from __future__ import annotations

import itertools
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    jaccard_score,
    matthews_corrcoef,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from estimate_annotation_ceiling import (  # noqa: E402
    BINARY_DATASETS,
    DATASETS,
    annotation_type,
    find_overlaps,
    flatten_matched_labels,
    labels_in_common_direction,
    load_dataset_records,
    threshold_continuous,
)

RESULTS = ROOT / "results"
CEILING_DIR = RESULTS / "annotation_ceiling"
HEADROOM_DIR = RESULTS / "normalized_headroom"
UDONPRED_DIR = ROOT / "UdonPred"
OUT_DIR = RESULTS / "slide_enhanced_ceiling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ceiling_summary = pd.read_csv(CEILING_DIR / "annotation_ceiling_summary.csv")
overlap_details = pd.read_csv(CEILING_DIR / "overlap_details.csv")
udon_matrix = pd.read_csv(RESULTS / "udonpred_matrix" / "matrix.csv").set_index("train_dataset")
cell_status = pd.read_csv(HEADROOM_DIR / "cell_status.csv")

sns.set_theme(style="whitegrid", context="notebook")
DATASETS


## 1. Biological Annotation Families

The annotation sources distinguish several disorder-related concepts:

- NMR chemical-shift disorder: direct residue-level experimental disorder signal.
- Curated disorder: manually curated binary evidence from heterogeneous experiments.
- AlphaFold confidence proxy: low confidence often tracks disorder, but it is not direct experimental disorder.
- Flexibility/dynamics: B-values and related structural mobility measurements capture dynamics, not disorder directly.
- Derived soft disorder resources: useful disorder-like labels but not a single direct physical observable.

These families are used below to interpret pairwise ceilings biologically.


In [ ]:
ANNOTATION_FAMILY = {
    "trizod": "NMR chemical-shift disorder",
    "chezod": "NMR chemical-shift disorder",
    "softdis": "derived soft disorder",
    "atlas": "derived soft disorder",
    "plddt": "AlphaFold confidence proxy",
    "disprot": "curated binary disorder",
    "pdbflex": "structural flexibility/dynamics",
}

DIRECTNESS_RANK = {
    "trizod": 1,
    "chezod": 1,
    "disprot": 2,
    "softdis": 3,
    "atlas": 3,
    "plddt": 4,
    "pdbflex": 5,
}

FAMILY_DESCRIPTION = {
    "NMR chemical-shift disorder": "Closest to direct residue-level physical disorder evidence.",
    "curated binary disorder": "Manually curated disorder evidence, binary and function-aware but less continuous.",
    "derived soft disorder": "Derived disorder-like targets that may combine or approximate multiple signals.",
    "AlphaFold confidence proxy": "Low-confidence structure prediction used as a disorder-like proxy, not a direct experiment.",
    "structural flexibility/dynamics": "B-factor/flexibility-like signal; related to motion, not identical to IDR biology.",
}

family_table = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "annotation_family": ANNOTATION_FAMILY[dataset],
            "directness_rank": DIRECTNESS_RANK[dataset],
            "lecture_based_meaning": FAMILY_DESCRIPTION[ANNOTATION_FAMILY[dataset]],
        }
        for dataset in DATASETS
    ]
)
display(family_table)
family_table.to_csv(OUT_DIR / "annotation_family_table.csv", index=False)


## 2. Primary Ceiling Table With Biological Compatibility

For continuous-continuous pairs, the primary ceiling metric is Spearman correlation. For pairs involving DisProt, the primary ceiling metric is AUROC when a continuous dataset is compared to the binary DisProt labels. This mirrors the original UdonPred evaluation style.

The new column `expected_biological_similarity` encodes a annotation-family expectation before looking at the numbers. This lets us ask whether the observed ceiling matches the biological interpretation from the annotation biology.


In [ ]:
PRIMARY_METRIC = {
    "continuous-continuous": "spearman",
    "continuous-binary": "auroc",
    "binary-continuous": "auroc",
    "binary-binary": "mcc",
}

def pair_key(a: str, b: str) -> tuple[str, str]:
    return tuple(sorted((a, b)))

HIGH_EXPECTED = {
    pair_key("trizod", "chezod"),
    pair_key("softdis", "plddt"),
    pair_key("softdis", "disprot"),
    pair_key("chezod", "plddt"),
}
MEDIUM_EXPECTED = {
    pair_key("trizod", "softdis"),
    pair_key("trizod", "plddt"),
    pair_key("chezod", "softdis"),
    pair_key("atlas", "softdis"),
    pair_key("atlas", "plddt"),
    pair_key("atlas", "chezod"),
}
LOW_EXPECTED = {
    pair_key("pdbflex", "trizod"),
    pair_key("pdbflex", "chezod"),
    pair_key("pdbflex", "softdis"),
    pair_key("pdbflex", "atlas"),
    pair_key("pdbflex", "plddt"),
    pair_key("pdbflex", "disprot"),
}

EXPECTATION_TO_SCORE = {
    "high": 3,
    "medium": 2,
    "low": 1,
    "unknown/no-overlap": np.nan,
}

def expected_similarity(a: str, b: str) -> str:
    key = pair_key(a, b)
    if key in HIGH_EXPECTED:
        return "high"
    if key in MEDIUM_EXPECTED:
        return "medium"
    if key in LOW_EXPECTED:
        return "low"
    return "unknown/no-overlap"

def overlap_confidence(n_residues: float) -> str:
    if pd.isna(n_residues) or n_residues <= 0:
        return "no estimate"
    if n_residues > 10_000:
        return "high"
    if n_residues >= 1_000:
        return "medium"
    if n_residues >= 300:
        return "low"
    return "very low"

def observed_strength(value: float, metric: str) -> str:
    if pd.isna(value):
        return "no estimate"
    if metric in {"spearman", "pearson", "mcc"}:
        abs_value = abs(value)
        if abs_value >= 0.60:
            return "high"
        if abs_value >= 0.30:
            return "medium"
        return "low"
    if metric in {"auroc", "average_precision"}:
        if value >= 0.80:
            return "high"
        if value >= 0.65:
            return "medium"
        return "low"
    return "not classified"

primary_rows = []
for (dataset_a, dataset_b), pair_df in ceiling_summary.groupby(["dataset_a", "dataset_b"]):
    annotation_key = f"{pair_df.iloc[0]['annotation_type_a']}-{pair_df.iloc[0]['annotation_type_b']}"
    metric = PRIMARY_METRIC.get(annotation_key)
    if metric is None:
        continue
    metric_row = pair_df[pair_df["metric"] == metric]
    if metric_row.empty:
        # Keep no-overlap rows visible.
        metric_row = pair_df[pair_df["metric"] == "no_overlap"]
    if metric_row.empty:
        continue
    row = metric_row.iloc[0].to_dict()
    row["primary_metric"] = metric
    row["family_a"] = ANNOTATION_FAMILY[dataset_a]
    row["family_b"] = ANNOTATION_FAMILY[dataset_b]
    row["expected_biological_similarity"] = expected_similarity(dataset_a, dataset_b)
    row["expected_similarity_score"] = EXPECTATION_TO_SCORE[row["expected_biological_similarity"]]
    row["overlap_confidence"] = overlap_confidence(row["n_residues_compared"])
    row["observed_strength"] = observed_strength(row["value"], metric)
    row["pair"] = f"{dataset_a} vs {dataset_b}"
    primary_rows.append(row)

primary_ceiling = pd.DataFrame(primary_rows).sort_values(
    ["overlap_confidence", "value"], ascending=[True, False]
)
display(primary_ceiling[[
    "pair", "family_a", "family_b", "metric", "value", "n_residues_compared",
    "overlap_confidence", "expected_biological_similarity", "observed_strength", "notes"
]])
primary_ceiling.to_csv(OUT_DIR / "primary_ceiling_biological_compatibility.csv", index=False)


## 3. Expected vs Observed Compatibility

This plot asks whether annotation-family expectations match observed annotation agreement. Pairs expected to be biologically similar should generally show stronger ceilings. The main counterexamples should be interpreted carefully, especially when overlap confidence is low.


In [ ]:
plot_df = primary_ceiling[primary_ceiling["value"].notna()].copy()
plot_df = plot_df[plot_df["expected_biological_similarity"].isin(["low", "medium", "high"])].copy()

similarity_order = ["low", "medium", "high"]
similarity_labels = {
    "low": "Low annotation-source\nsimilarity",
    "medium": "Medium annotation-source\nsimilarity",
    "high": "High annotation-source\nsimilarity",
}
plot_df["group_label"] = pd.Categorical(
    plot_df["expected_biological_similarity"].map(similarity_labels),
    categories=[similarity_labels[level] for level in similarity_order],
    ordered=True,
)

expected_summary = (
    plot_df.groupby("expected_biological_similarity", observed=False)
    .agg(
        n_pairs=("pair", "count"),
        median_observed_ceiling=("value", "median"),
        mean_observed_ceiling=("value", "mean"),
        median_residue_overlap=("n_residues_compared", "median"),
    )
    .reindex(similarity_order)
    .reset_index()
)
display(expected_summary)
expected_summary.to_csv(OUT_DIR / "expected_similarity_summary.csv", index=False)

bar_summary = (
    plot_df.groupby("group_label", observed=True)
    .agg(median=("value", "median"), n=("value", "size"))
    .reset_index()
)

palette = ["#df9a86", "#f1c973", "#91bf9e"]
point_color = "#3e3e3e"
fig, ax = plt.subplots(figsize=(11.8, 6.1))
x_positions = np.arange(len(bar_summary))
ax.bar(
    x_positions,
    bar_summary["median"],
    width=0.58,
    color=palette,
    edgecolor="none",
    zorder=1,
)

# Dot y-positions are the measured ceiling values. X offsets are fixed display-only
# offsets, sorted by value within each group, to avoid overplotting without implying
# another measured variable.
point_positions = {}
for idx, group in enumerate(similarity_order):
    group_values = (
        plot_df.loc[plot_df["expected_biological_similarity"].eq(group), ["pair", "value"]]
        .sort_values(["value", "pair"])
        .reset_index(drop=True)
    )
    offsets = np.linspace(-0.16, 0.16, len(group_values)) if len(group_values) > 1 else np.array([0.0])
    x_values = np.full(len(group_values), idx) + offsets
    ax.scatter(
        x_values,
        group_values["value"],
        s=62,
        facecolor="white",
        edgecolor=point_color,
        linewidth=1.4,
        zorder=3,
    )
    for pair, x_value, y_value in zip(group_values["pair"], x_values, group_values["value"]):
        point_positions[pair] = (float(x_value), float(y_value))

for idx, row in bar_summary.iterrows():
    median_value = row["median"]
    ax.text(
        idx,
        median_value + 0.09,
        f"median {median_value:.2f}",
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        color="#111111",
        zorder=4,
    )
    ax.text(
        idx,
        -0.088,
        f"n={int(row['n'])} pairs",
        ha="center",
        va="top",
        fontsize=10.5,
        color="#666666",
    )

callouts = [
    ("softdis vs pdbflex", "SoftDis-PDBFlex\n0.08 over 123k residues\nflexibility != disorder", (0.18, 0.34)),
    ("softdis vs disprot", "SoftDis-DisProt\nAUROC 0.93\nshared disorder signal", (1.63, 0.95)),
    ("chezod vs plddt", "CheZOD-pLDDT\n0.69\npLDDT tracks disorder proxy", (2.25, 0.46)),
]
for pair, text, xytext in callouts:
    ax.annotate(
        text,
        xy=point_positions[pair],
        xytext=xytext,
        textcoords="data",
        ha="center",
        va="center",
        fontsize=10.5,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#bdbdbd", lw=1.0),
        arrowprops=dict(arrowstyle="->", color="#555555", lw=1.1, shrinkA=4, shrinkB=5),
        zorder=5,
    )

ax.set_title("expected vs observed compatibility", fontsize=17, pad=14)
ax.set_ylabel("Observed annotation ceiling\n(Spearman or AUROC)", fontsize=12.5)
ax.set_xlabel("Biological similarity between annotation sources", fontsize=12.5)
ax.set_xticks(x_positions)
ax.set_xticklabels(bar_summary["group_label"], fontsize=11.5)
ax.set_ylim(-0.13, 1.05)
ax.set_xlim(-0.52, 2.72)
ax.grid(axis="y", color="#d7d7d7", linewidth=0.8)
ax.grid(axis="x", visible=False)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)
fig.text(
    0.5,
    0.018,
    "Dots are individual dataset pairs; bars show median ceiling per annotation-source similarity group. Horizontal offsets only prevent overlap.",
    ha="center",
    va="bottom",
    fontsize=10.5,
    color="#555555",
)
fig.tight_layout(rect=(0.075, 0.075, 0.99, 0.97))
for figure_name in [
    "expected_vs_observed_compatibility.png",
    "expected_vs_observed_compatibility_slide.png",
    "expected_vs_observed_compatibility_clear.png",
]:
    fig.savefig(OUT_DIR / figure_name, dpi=220, bbox_inches="tight")
plt.show()


## 4. Annotation-Family Agreement Summary

This summarizes ceilings by biological source family rather than only by dataset pair. It directly tests the measurement-aware idea that NMR disorder, confidence proxies, curated disorder, and flexibility are related but not identical.


In [ ]:
def family_pair(row: pd.Series) -> str:
    families = sorted([row["family_a"], row["family_b"]])
    return " <-> ".join(families)

family_df = plot_df.copy()
family_df["family_pair"] = family_df.apply(family_pair, axis=1)
family_summary = (
    family_df.groupby("family_pair")
    .agg(
        n_pairs=("pair", "count"),
        median_ceiling=("value", "median"),
        mean_ceiling=("value", "mean"),
        max_ceiling=("value", "max"),
        total_residues=("n_residues_compared", "sum"),
        example_pairs=("pair", lambda s: "; ".join(s.head(4))),
    )
    .sort_values("median_ceiling", ascending=False)
    .reset_index()
)
display(family_summary)
family_summary.to_csv(OUT_DIR / "annotation_family_agreement_summary.csv", index=False)

plt.figure(figsize=(10, max(4, 0.35 * len(family_summary))))
sns.barplot(data=family_summary, y="family_pair", x="median_ceiling", color="#4C78A8")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Median observed primary ceiling")
plt.ylabel("Annotation-family pair")
plt.title("Ceiling Agreement by Biological Annotation Family")
plt.tight_layout()
plt.savefig(OUT_DIR / "family_pair_ceiling_summary.png", dpi=200)
plt.show()


## 5. Same-Concept vs Cross-Concept Summary

The biological annotation model motivates a distinction between sources that should measure a related disorder concept and sources that measure a different biological axis. Here, PDBFlex/flexibility comparisons are treated as a different-concept contrast because flexibility/dynamics should be distinguished B-values/flexibility from disorder.


In [ ]:
def concept_relation(a: str, b: str) -> str:
    if "pdbflex" in {a, b}:
        return "different concept: flexibility vs disorder/proxy"
    if ANNOTATION_FAMILY[a] == ANNOTATION_FAMILY[b]:
        return "same annotation family"
    if expected_similarity(a, b) in {"high", "medium"}:
        return "related disorder concept"
    return "unknown or weakly related"

concept_df = plot_df.copy()
concept_df["concept_relation"] = concept_df.apply(lambda r: concept_relation(r["dataset_a"], r["dataset_b"]), axis=1)
concept_summary = (
    concept_df.groupby("concept_relation")
    .agg(
        n_pairs=("pair", "count"),
        median_ceiling=("value", "median"),
        mean_ceiling=("value", "mean"),
        median_overlap=("n_residues_compared", "median"),
        pairs=("pair", lambda s: "; ".join(s)),
    )
    .sort_values("median_ceiling", ascending=False)
    .reset_index()
)
display(concept_summary)
concept_summary.to_csv(OUT_DIR / "same_vs_cross_concept_summary.csv", index=False)

plt.figure(figsize=(9, 5))
sns.boxplot(data=concept_df, x="value", y="concept_relation", color="#72B7B2")
sns.stripplot(data=concept_df, x="value", y="concept_relation", color="black", size=5)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Observed primary ceiling")
plt.ylabel("")
plt.title("Same/Related Disorder Concepts vs Different Biological Concepts")
plt.tight_layout()
plt.savefig(OUT_DIR / "same_vs_cross_concept_ceiling.png", dpi=200)
plt.show()


## 6. Direction-Specific Transfer Interpretation

The annotation ceiling is symmetric, but model transfer is directional: training on PDBFlex and testing on pLDDT asks a different question than training on pLDDT and testing on PDBFlex. This table adds biological hypotheses to the UdonPred transfer matrix.


In [ ]:
def udon_column_for_test_dataset(test_dataset: str) -> list[str]:
    if test_dataset == "disprot":
        return ["disprot\n(AP)", "disprot\n(AUROC)"]
    return [test_dataset]

def transfer_hypothesis(train_dataset: str, test_dataset: str) -> str:
    if train_dataset == test_dataset:
        return "same annotation definition; should be the most concept-aligned transfer"
    train_family = ANNOTATION_FAMILY[train_dataset]
    test_family = ANNOTATION_FAMILY[test_dataset]
    if "structural flexibility/dynamics" in {train_family, test_family}:
        return "tests whether flexibility/dynamics can substitute for disorder; lecture predicts weak transfer"
    if {train_family, test_family} == {"NMR chemical-shift disorder", "AlphaFold confidence proxy"}:
        return "tests whether direct NMR disorder aligns with AlphaFold low-confidence disorder proxy"
    if "curated binary disorder" in {train_family, test_family}:
        return "tests whether continuous/proxy disorder ranks curated DisProt-positive residues"
    if expected_similarity(train_dataset, test_dataset) in {"high", "medium"}:
        return "tests transfer between related disorder-like concepts"
    return "tests transfer between weakly characterized or weakly overlapping annotation concepts"

rows = []
for train_dataset in DATASETS:
    for test_dataset in DATASETS:
        for test_metric in udon_column_for_test_dataset(test_dataset):
            if train_dataset not in udon_matrix.index or test_metric not in udon_matrix.columns:
                continue
            status_rows = cell_status[
                (cell_status["train_dataset"] == train_dataset) &
                (cell_status["test_metric"] == test_metric)
            ]
            status = status_rows.iloc[0].to_dict() if not status_rows.empty else {}
            rows.append({
                "train_dataset": train_dataset,
                "test_metric": test_metric,
                "test_dataset": test_dataset,
                "train_family": ANNOTATION_FAMILY[train_dataset],
                "test_family": ANNOTATION_FAMILY[test_dataset],
                "biological_hypothesis": transfer_hypothesis(train_dataset, test_dataset),
                "expected_biological_similarity": expected_similarity(train_dataset, test_dataset) if train_dataset != test_dataset else "same dataset",
                "udon_score": udon_matrix.loc[train_dataset, test_metric],
                "annotation_ceiling": status.get("annotation_ceiling", np.nan),
                "normalized_headroom": status.get("normalized_headroom", np.nan),
                "status": status.get("status", "not available"),
                "ceiling_pair": status.get("ceiling_pair", ""),
                "n_residues_compared": status.get("n_residues_compared", np.nan),
            })

direction_table = pd.DataFrame(rows)
display(direction_table.sort_values(["test_dataset", "udon_score"], ascending=[True, False]).head(20))
direction_table.to_csv(OUT_DIR / "direction_specific_transfer_interpretation.csv", index=False)


## 7. Reconstruct Matched Labels for Threshold and Region Analyses

The next analyses need residue-level labels, not only the ceiling summary. We reuse the exact-match overlap logic and label-direction convention from the existing ceiling script:

- CheZOD is negated so larger means more disorder.
- pLDDT is converted to disorder-like score with `1 - pLDDT / 100`.
- DisProt remains binary.


In [ ]:
records_by_dataset = {
    dataset: load_dataset_records(UDONPRED_DIR, dataset)
    for dataset in DATASETS
}

matched_cache = {}
for dataset_a, dataset_b in itertools.combinations(DATASETS, 2):
    matches = find_overlaps(records_by_dataset[dataset_a], records_by_dataset[dataset_b], min_residues=30)
    labels_a, labels_b = flatten_matched_labels(matches, dataset_a, dataset_b)
    matched_cache[(dataset_a, dataset_b)] = {
        "matches": matches,
        "labels_a": labels_a,
        "labels_b": labels_b,
        "n_proteins": len(matches),
        "n_residues": len(labels_a),
    }

matched_overview = pd.DataFrame([
    {
        "dataset_a": a,
        "dataset_b": b,
        "n_proteins_overlap": item["n_proteins"],
        "n_residues_compared": item["n_residues"],
    }
    for (a, b), item in matched_cache.items()
]).sort_values("n_residues_compared", ascending=False)
display(matched_overview.head(12))
matched_overview.to_csv(OUT_DIR / "reconstructed_overlap_overview.csv", index=False)


## 8. Threshold Sensitivity for DisProt Comparisons

AUROC is threshold-free, but biological binary interpretation requires a threshold. The project description notes that thresholds such as G = 0.5 matter. Here we vary the continuous-score threshold and recompute binary agreement with DisProt using F1, MCC, balanced accuracy, and Jaccard.


In [ ]:
thresholds = np.round(np.arange(0.1, 0.91, 0.05), 2)
threshold_rows = []

for (dataset_a, dataset_b), item in matched_cache.items():
    if "disprot" not in {dataset_a, dataset_b}:
        continue
    if item["n_residues"] == 0:
        continue

    labels_a = item["labels_a"]
    labels_b = item["labels_b"]
    if dataset_a == "disprot":
        binary = labels_a.astype(int)
        continuous = labels_b
        continuous_dataset = dataset_b
    else:
        continuous = labels_a
        binary = labels_b.astype(int)
        continuous_dataset = dataset_a

    if len(np.unique(binary)) < 2:
        continue

    for threshold in thresholds:
        pred = (continuous >= threshold).astype(int)
        threshold_rows.append({
            "continuous_dataset": continuous_dataset,
            "binary_dataset": "disprot",
            "threshold": threshold,
            "n_residues": len(binary),
            "positive_rate_true": float(binary.mean()),
            "positive_rate_pred": float(pred.mean()),
            "f1": f1_score(binary, pred, zero_division=0),
            "mcc": matthews_corrcoef(binary, pred) if len(np.unique(pred)) > 1 else np.nan,
            "balanced_accuracy": balanced_accuracy_score(binary, pred),
            "jaccard": jaccard_score(binary, pred, zero_division=0),
        })

threshold_sensitivity = pd.DataFrame(threshold_rows)
if threshold_sensitivity.empty:
    print("No DisProt overlap with two binary classes was available for threshold sensitivity.")
else:
    display(threshold_sensitivity.sort_values(["continuous_dataset", "f1"], ascending=[True, False]).groupby("continuous_dataset").head(3))
    threshold_sensitivity.to_csv(OUT_DIR / "disprot_threshold_sensitivity.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    sns.lineplot(data=threshold_sensitivity, x="threshold", y="f1", hue="continuous_dataset", marker="o", ax=axes[0])
    axes[0].set_title("DisProt Threshold Sensitivity: F1")
    axes[0].set_ylabel("F1")
    sns.lineplot(data=threshold_sensitivity, x="threshold", y="mcc", hue="continuous_dataset", marker="o", ax=axes[1], legend=False)
    axes[1].set_title("DisProt Threshold Sensitivity: MCC")
    axes[1].set_ylabel("MCC")
    for ax in axes:
        ax.set_xlabel("Continuous disorder threshold")
        ax.axhline(0, color="black", linewidth=0.8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "disprot_threshold_sensitivity.png", dpi=200)
    plt.show()


## 9. Region-Level Agreement

Residue-wise agreement can be harsh. The annotation biology includes disorder regions, long loops/NORS, molecular recognition elements, and other region-level concepts. Two datasets may disagree at exact residue boundaries but still identify the same broad disordered segment.

This section thresholds continuous scores, merges consecutive positive residues into regions, and computes region-level overlap. The implementation uses residue-level binary Jaccard plus segment-level precision/recall/F1 based on whether predicted and reference regions overlap by at least one residue.


In [ ]:
def binarize_labels(labels: np.ndarray, dataset: str, threshold: float = 0.5) -> np.ndarray:
    if dataset in BINARY_DATASETS:
        return labels.astype(int)
    return (labels >= threshold).astype(int)

def binary_regions(binary: np.ndarray) -> list[tuple[int, int]]:
    regions = []
    start = None
    for i, value in enumerate(binary):
        if value == 1 and start is None:
            start = i
        elif value == 0 and start is not None:
            regions.append((start, i - 1))
            start = None
    if start is not None:
        regions.append((start, len(binary) - 1))
    return regions

def regions_overlap(a: tuple[int, int], b: tuple[int, int]) -> bool:
    return max(a[0], b[0]) <= min(a[1], b[1])

def region_overlap_metrics(binary_a: np.ndarray, binary_b: np.ndarray) -> dict[str, float]:
    regions_a = binary_regions(binary_a)
    regions_b = binary_regions(binary_b)
    if not regions_a and not regions_b:
        return {
            "n_regions_a": 0,
            "n_regions_b": 0,
            "region_precision": 1.0,
            "region_recall": 1.0,
            "region_f1": 1.0,
            "residue_jaccard": 1.0,
        }
    matched_a = sum(any(regions_overlap(a, b) for b in regions_b) for a in regions_a)
    matched_b = sum(any(regions_overlap(a, b) for a in regions_a) for b in regions_b)
    precision = matched_a / len(regions_a) if regions_a else 0.0
    recall = matched_b / len(regions_b) if regions_b else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "n_regions_a": len(regions_a),
        "n_regions_b": len(regions_b),
        "region_precision": precision,
        "region_recall": recall,
        "region_f1": f1,
        "residue_jaccard": jaccard_score(binary_b, binary_a, zero_division=0),
    }

region_rows = []
for (dataset_a, dataset_b), item in matched_cache.items():
    if item["n_residues"] == 0:
        continue
    binary_a = binarize_labels(item["labels_a"], dataset_a)
    binary_b = binarize_labels(item["labels_b"], dataset_b)
    metrics = region_overlap_metrics(binary_a, binary_b)
    region_rows.append({
        "dataset_a": dataset_a,
        "dataset_b": dataset_b,
        "pair": f"{dataset_a} vs {dataset_b}",
        "family_a": ANNOTATION_FAMILY[dataset_a],
        "family_b": ANNOTATION_FAMILY[dataset_b],
        "n_residues": item["n_residues"],
        "positive_rate_a": float(binary_a.mean()) if len(binary_a) else np.nan,
        "positive_rate_b": float(binary_b.mean()) if len(binary_b) else np.nan,
        **metrics,
    })

region_agreement = pd.DataFrame(region_rows).sort_values("region_f1", ascending=False)
display(region_agreement)
region_agreement.to_csv(OUT_DIR / "region_level_agreement.csv", index=False)

plt.figure(figsize=(10, max(4, 0.35 * len(region_agreement))))
sns.barplot(data=region_agreement, y="pair", x="region_f1", hue="family_a", dodge=False)
plt.xlabel("Region-level F1 after thresholding")
plt.ylabel("Dataset pair")
plt.title("Region-Level Agreement Between Annotation Sources")
plt.legend(title="Family A", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUT_DIR / "region_level_agreement.png", dpi=200)
plt.show()


## 10. PDBFlex Flexibility-Control Analysis

The lecture explicitly states that B-factors/flexibility capture aspects of protein dynamics, not disorder directly. PDBFlex is therefore useful as a contrast/control: if the ceiling analysis is biologically meaningful, PDBFlex should show weaker agreement with NMR disorder, AlphaFold-confidence disorder proxies, and curated disorder than the more disorder-like datasets do.


In [ ]:
pdbflex_control = primary_ceiling[
    (primary_ceiling["dataset_a"] == "pdbflex") | (primary_ceiling["dataset_b"] == "pdbflex")
].copy()
pdbflex_control = pdbflex_control.sort_values("n_residues_compared", ascending=False)
display(pdbflex_control[[
    "pair", "metric", "value", "n_residues_compared", "overlap_confidence",
    "expected_biological_similarity", "observed_strength", "notes"
]])
pdbflex_control.to_csv(OUT_DIR / "pdbflex_flexibility_control.csv", index=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=pdbflex_control, x="value", y="pair", color="#F58518")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Observed primary ceiling")
plt.ylabel("PDBFlex comparison")
plt.title("PDBFlex as Flexibility-Not-Disorder Control")
plt.tight_layout()
plt.savefig(OUT_DIR / "pdbflex_control.png", dpi=200)
plt.show()


## 11. Biological Claim Table

This final table converts the analyses into concise claims that can be used in a report or presentation. Each claim links numerical evidence to a lecture concept and a biological interpretation.


In [ ]:
def get_primary_value(a: str, b: str) -> tuple[float, float, str]:
    row = primary_ceiling[
        ((primary_ceiling["dataset_a"] == a) & (primary_ceiling["dataset_b"] == b)) |
        ((primary_ceiling["dataset_a"] == b) & (primary_ceiling["dataset_b"] == a))
    ]
    if row.empty:
        return np.nan, np.nan, "not estimated"
    r = row.iloc[0]
    return r["value"], r["n_residues_compared"], r["metric"]

soft_pdb_val, soft_pdb_n, soft_pdb_metric = get_primary_value("softdis", "pdbflex")
pdb_plddt_val, pdb_plddt_n, pdb_plddt_metric = get_primary_value("pdbflex", "plddt")
chez_plddt_val, chez_plddt_n, chez_plddt_metric = get_primary_value("chezod", "plddt")
soft_disprot_val, soft_disprot_n, soft_disprot_metric = get_primary_value("softdis", "disprot")
tri_chez_val, tri_chez_n, tri_chez_metric = get_primary_value("trizod", "chezod")

claim_rows = [
    {
        "finding": "PDBFlex is a distinct flexibility target, not a generic disorder target.",
        "numerical_evidence": f"SoftDis vs PDBFlex {soft_pdb_metric} = {soft_pdb_val:.3f} over {soft_pdb_n:.0f} residues; PDBFlex vs pLDDT {pdb_plddt_metric} = {pdb_plddt_val:.3f}.",
        "lecture_concept": "B-values/flexibility capture dynamics, not disorder directly.",
        "biological_interpretation": "Flexibility labels should be modeled and interpreted separately from IDR labels.",
    },
    {
        "finding": "pLDDT aligns with several disorder-like sources but remains a proxy.",
        "numerical_evidence": f"CheZOD vs pLDDT {chez_plddt_metric} = {chez_plddt_val:.3f} over {chez_plddt_n:.0f} residues.",
        "lecture_concept": "AlphaFold confidence and pLM disorder predictions often correlate, but confidence is not direct experiment.",
        "biological_interpretation": "Low AlphaFold confidence can be useful for disorder prediction, but should not replace NMR or curated evidence.",
    },
    {
        "finding": "SoftDis is compatible with curated DisProt disorder on overlapping proteins.",
        "numerical_evidence": f"SoftDis vs DisProt {soft_disprot_metric} = {soft_disprot_val:.3f} over {soft_disprot_n:.0f} residues.",
        "lecture_concept": "Different disorder methods can recover a shared signal even with heterogeneous labels.",
        "biological_interpretation": "SoftDis likely captures a broad disorder signal relevant to curated functional disorder.",
    },
    {
        "finding": "NMR-derived datasets share signal but current overlap is too small for a stable ceiling.",
        "numerical_evidence": f"TriZOD vs CheZOD {tri_chez_metric} = {tri_chez_val:.3f}, but only {tri_chez_n:.0f} residues overlap.",
        "lecture_concept": "Chemical shifts provide continuous direct disorder evidence; dataset construction and normalization still matter.",
        "biological_interpretation": "More matched NMR proteins are needed before claiming a robust NMR-to-NMR upper bound.",
    },
    {
        "finding": "Ceiling analysis should guide interpretation, not be treated as a hard universal bound.",
        "numerical_evidence": "Several ceiling estimates have tiny overlaps or no overlap, while UdonPred is evaluated on full test sets.",
        "lecture_concept": "Proper method comparison requires same measure, same dataset, and careful accounting for experimental error.",
        "biological_interpretation": "Scores above an overlap-based ceiling usually indicate non-comparability or sampling limits, not super-human annotation performance.",
    },
]

biological_claims = pd.DataFrame(claim_rows)
display(biological_claims)
biological_claims.to_csv(OUT_DIR / "biological_claim_table.csv", index=False)


## 12. MMseqs Homology-Based Ceiling Analysis

The analyses above use exact ID or exact sequence overlap. The existing ceiling script also supports MMseqs2 local-alignment comparisons, which answer a different biological question.

- Exact overlap asks: *do two datasets agree on the same protein and same residue positions?*
- MMseqs overlap asks: *does the annotation concept transfer across highly similar homologs and aligned positions?*

This distinction matters because the annotation-transfer framework includes homology-based inference: annotation transfer is reliable only when the biological property is conserved across sequence-similar proteins. For disorder, this is not guaranteed. Some disorder regions are conserved, but others are context-dependent, composition-driven, or shifted in boundaries. Therefore MMseqs results should be interpreted as a homologous-annotation-transfer ceiling, not as a pure experimental reproducibility ceiling.


In [ ]:
MMSEQS_DIR = RESULTS / "annotation_ceiling_mmseqs"
mmseqs_summary_path = MMSEQS_DIR / "annotation_ceiling_summary.csv"
if not mmseqs_summary_path.exists():
    raise FileNotFoundError(
        f"Missing {mmseqs_summary_path}. Run: python scripts/estimate_annotation_ceiling.py "
        "--use-mmseqs --output-dir results/annotation_ceiling_mmseqs"
    )

mmseqs_summary = pd.read_csv(mmseqs_summary_path)
mmseqs_summary["value"] = pd.to_numeric(mmseqs_summary["value"], errors="coerce")
mmseqs_summary["min_identity"] = pd.to_numeric(mmseqs_summary["min_identity"], errors="coerce")
mmseqs_summary["n_residues_compared"] = pd.to_numeric(mmseqs_summary["n_residues_compared"], errors="coerce")
mmseqs_summary["n_proteins_overlap"] = pd.to_numeric(mmseqs_summary["n_proteins_overlap"], errors="coerce")

mmseqs_primary_rows = []
for (dataset_a, dataset_b, comparison_level), pair_df in mmseqs_summary.groupby(["dataset_a", "dataset_b", "comparison_level"]):
    annotation_key = f"{pair_df.iloc[0]['annotation_type_a']}-{pair_df.iloc[0]['annotation_type_b']}"
    metric = PRIMARY_METRIC.get(annotation_key)
    if metric is None:
        continue
    metric_row = pair_df[pair_df["metric"] == metric]
    if metric_row.empty:
        metric_row = pair_df[pair_df["metric"] == "no_overlap"]
    if metric_row.empty:
        continue
    row = metric_row.iloc[0].to_dict()
    row["primary_metric"] = metric
    row["family_a"] = ANNOTATION_FAMILY[dataset_a]
    row["family_b"] = ANNOTATION_FAMILY[dataset_b]
    row["expected_biological_similarity"] = expected_similarity(dataset_a, dataset_b)
    row["overlap_confidence"] = overlap_confidence(row["n_residues_compared"])
    row["observed_strength"] = observed_strength(row["value"], metric)
    row["concept_relation"] = concept_relation(dataset_a, dataset_b)
    row["pair"] = f"{dataset_a} vs {dataset_b}"
    row["is_mmseqs"] = str(comparison_level).startswith("mmseqs_")
    mmseqs_primary_rows.append(row)

mmseqs_primary = pd.DataFrame(mmseqs_primary_rows)
mmseqs_primary = mmseqs_primary.sort_values(["comparison_level", "pair"])
display(mmseqs_primary[[
    "comparison_level", "pair", "primary_metric", "value", "min_identity",
    "n_proteins_overlap", "n_residues_compared", "expected_biological_similarity",
    "concept_relation", "observed_strength"
]].head(30))
mmseqs_primary.to_csv(OUT_DIR / "mmseqs_primary_biological_compatibility.csv", index=False)


### Identity-Threshold Trend

If disorder annotations transfer through homology, the ceiling should remain high as we include lower-identity homologs. If the property is not conserved, or if region boundaries shift, agreement should drop or become unstable as identity decreases. This trend is biologically more informative than a single MMseqs number.


In [ ]:
mmseqs_only = mmseqs_primary[mmseqs_primary["is_mmseqs"] & mmseqs_primary["value"].notna()].copy()
mmseqs_only["identity_threshold"] = mmseqs_only["min_identity"]

identity_summary = (
    mmseqs_only.groupby("identity_threshold")
    .agg(
        n_pairs=("pair", "count"),
        median_primary_ceiling=("value", "median"),
        mean_primary_ceiling=("value", "mean"),
        total_residues=("n_residues_compared", "sum"),
        median_residues_per_pair=("n_residues_compared", "median"),
    )
    .sort_index(ascending=False)
    .reset_index()
)
display(identity_summary)
identity_summary.to_csv(OUT_DIR / "mmseqs_identity_threshold_summary.csv", index=False)

plt.figure(figsize=(12, 7))
sns.lineplot(
    data=mmseqs_only,
    x="identity_threshold",
    y="value",
    hue="pair",
    style="primary_metric",
    marker="o",
)
plt.gca().invert_xaxis()
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Minimum MMseqs identity (%)")
plt.ylabel("Primary homologous ceiling")
plt.title("MMseqs Homologous Annotation Agreement by Identity Threshold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Pair / metric")
plt.tight_layout()
plt.savefig(OUT_DIR / "mmseqs_primary_agreement_by_identity.png", dpi=200)
plt.show()


### Exact vs MMseqs-80 Comparison

The 80% identity threshold is the broadest homologous comparison currently computed. It increases coverage but also changes the biological interpretation: we are no longer comparing the same protein, but similar aligned proteins. Large changes between exact and MMseqs-80 indicate that homolog inclusion changes the sampled biology.


In [ ]:
exact_primary = mmseqs_primary[mmseqs_primary["comparison_level"] == "exact"][[
    "dataset_a", "dataset_b", "pair", "primary_metric", "value", "n_residues_compared", "n_proteins_overlap"
]].rename(columns={
    "value": "exact_value",
    "n_residues_compared": "exact_residues",
    "n_proteins_overlap": "exact_proteins",
})
mm80_primary = mmseqs_primary[mmseqs_primary["comparison_level"] == "mmseqs_80"][[
    "dataset_a", "dataset_b", "value", "n_residues_compared", "n_proteins_overlap",
    "expected_biological_similarity", "concept_relation"
]].rename(columns={
    "value": "mmseqs80_value",
    "n_residues_compared": "mmseqs80_residues",
    "n_proteins_overlap": "mmseqs80_proteins",
})
exact_vs_mm80 = exact_primary.merge(mm80_primary, on=["dataset_a", "dataset_b"], how="outer")
exact_vs_mm80["pair"] = exact_vs_mm80["pair"].fillna(exact_vs_mm80["dataset_a"] + " vs " + exact_vs_mm80["dataset_b"])
exact_vs_mm80["delta_mmseqs80_minus_exact"] = exact_vs_mm80["mmseqs80_value"] - exact_vs_mm80["exact_value"]
exact_vs_mm80["residue_gain"] = exact_vs_mm80["mmseqs80_residues"] - exact_vs_mm80["exact_residues"]
exact_vs_mm80 = exact_vs_mm80.sort_values("delta_mmseqs80_minus_exact", ascending=False)
display(exact_vs_mm80[[
    "pair", "primary_metric", "exact_value", "mmseqs80_value", "delta_mmseqs80_minus_exact",
    "exact_residues", "mmseqs80_residues", "residue_gain", "expected_biological_similarity", "concept_relation"
]])
exact_vs_mm80.to_csv(OUT_DIR / "exact_vs_mmseqs80_primary_ceiling.csv", index=False)

plt.figure(figsize=(9, 6))
plot_delta = exact_vs_mm80[exact_vs_mm80["delta_mmseqs80_minus_exact"].notna()].copy()
sns.barplot(data=plot_delta, y="pair", x="delta_mmseqs80_minus_exact", hue="concept_relation", dodge=False)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("MMseqs-80 primary ceiling minus exact primary ceiling")
plt.ylabel("Dataset pair")
plt.title("How Homologous Matching Changes Estimated Annotation Agreement")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Concept relation")
plt.tight_layout()
plt.savefig(OUT_DIR / "exact_vs_mmseqs80_delta.png", dpi=200)
plt.show()


### Family-Level Interpretation of MMseqs Results

This summarizes the broadest homologous comparison (`mmseqs_80`) by biological family. The important question is whether the original exact-overlap conclusions survive after increasing overlap through homologous matches.


In [ ]:
mm80 = mmseqs_primary[(mmseqs_primary["comparison_level"] == "mmseqs_80") & mmseqs_primary["value"].notna()].copy()
mm80_family_summary = (
    mm80.groupby("concept_relation")
    .agg(
        n_pairs=("pair", "count"),
        median_primary_ceiling=("value", "median"),
        mean_primary_ceiling=("value", "mean"),
        total_residues=("n_residues_compared", "sum"),
        example_pairs=("pair", lambda s: "; ".join(s.head(6))),
    )
    .sort_values("median_primary_ceiling", ascending=False)
    .reset_index()
)
display(mm80_family_summary)
mm80_family_summary.to_csv(OUT_DIR / "mmseqs80_concept_relation_summary.csv", index=False)

pdbflex_mmseqs = mm80[(mm80["dataset_a"] == "pdbflex") | (mm80["dataset_b"] == "pdbflex")].copy()
pdbflex_mmseqs = pdbflex_mmseqs.sort_values("value", ascending=False)
display(pdbflex_mmseqs[[
    "pair", "primary_metric", "value", "n_residues_compared", "n_proteins_overlap",
    "expected_biological_similarity", "observed_strength"
]])
pdbflex_mmseqs.to_csv(OUT_DIR / "mmseqs80_pdbflex_control.csv", index=False)


### MMseqs Biological Interpretation

The MMseqs analysis should be read as a homology-based annotation-transfer experiment. It increases coverage and reveals additional biologically plausible relationships, but it is less strict than exact same-protein overlap.

The most important pattern is that high-similarity homologs can substantially increase apparent agreement for some pairs, especially `TriZOD` vs `CheZOD` and `CheZOD` vs `Atlas`. That suggests some disorder annotations are conserved across close homologs or that the newly included homologs are more representative than the tiny exact-overlap sample. However, this does not prove experimental reproducibility; it proves homologous annotation compatibility under high sequence identity and high coverage.

The second important pattern is that the PDBFlex conclusion remains mostly intact. Even with MMseqs-80, PDBFlex comparisons remain relatively weak compared with the strongest disorder-like pairs. `SoftDis` vs `PDBFlex` rises only from 0.077 exact to 0.141 with MMseqs-80 despite more than 200,000 aligned residues. This supports the measurement-aware interpretation that flexibility/dynamics is related to, but biologically distinct from, intrinsic disorder.

The third important pattern is that pLDDT and DisProt become visible under MMseqs even when exact overlap was absent. `pLDDT` vs `DisProt` reaches high AUROC across homologous matches, supporting the idea that low AlphaFold confidence often marks regions related to curated disorder. But this is still a proxy/homology result: it should be reported separately from exact overlap.


## Overall Interpretation

The enhanced ceiling analysis supports the measurement-aware view that disorder prediction is limited by annotation heterogeneity. NMR chemical shifts, pLDDT confidence, curated DisProt labels, and PDBFlex flexibility all contain biologically meaningful information, but they are not interchangeable. The strongest added conclusion is the PDBFlex control result: flexibility/dynamics comparisons remain weak even with large overlap, so PDBFlex should not be interpreted as a generic disorder ceiling. Conversely, CheZOD/pLDDT and SoftDis/DisProt show that some sources do recover a shared disorder axis.

For modeling, this means model heads should be selected and interpreted according to the biological target they reproduce, not only according to raw metric values. For reporting, the relevant question becomes: *which disorder concept did the model learn, and how compatible is that concept with the target annotation?*
